# Subgraphs

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangGraph roadmap.

You will learn how to compose LangGraph applications by using compiled graphs as nodes inside parent graphs, with two patterns for different schema situations and multi-level nesting.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Pattern 1 — Call a Subgraph Inside a Node

When the subgraph has a different schema from the parent, call it inside a node function and manually transform state.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END, add_messages
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

class ChildState(TypedDict):
    query: str
    result: str

def child_research(state: ChildState) -> dict:
    response = llm.invoke([
        ("system", "You are a research assistant. Provide concise findings."),
        ("human", state["query"]),
    ])
    return {"result": response.content}

child_graph = StateGraph(ChildState)
child_graph.add_node("research", child_research)
child_graph.add_edge(START, "research")
child_graph.add_edge("research", END)
child_app = child_graph.compile()

print("Child subgraph compiled successfully")

In [ ]:
class ParentState(TypedDict):
    question: str
    answer: str
    messages: Annotated[list, add_messages]

def call_child(state: ParentState) -> dict:
    child_result = child_app.invoke({"query": state["question"]})
    return {"answer": child_result["result"]}

parent_graph = StateGraph(ParentState)
parent_graph.add_node("call_child", call_child)
parent_graph.add_edge(START, "call_child")
parent_graph.add_edge("call_child", END)
parent_app = parent_graph.compile()

result = parent_app.invoke({"question": "What is LangGraph?"})
print(f"Answer: {result['answer'][:200]}...")

## 4. Pattern 2 — Add a Compiled Graph as a Direct Node

When the subgraph shares the same state schema, add the compiled graph directly as a node.

In [ ]:
class SharedState(TypedDict):
    question: str
    analysis: str
    answer: str

def analyze(state: SharedState) -> dict:
    response = llm.invoke([
        ("system", "Analyze the question and provide key insights."),
        ("human", state["question"]),
    ])
    return {"analysis": response.content}

def summarize(state: SharedState) -> dict:
    response = llm.invoke([
        ("system", "Summarize the analysis into a concise answer."),
        ("human", state["analysis"]),
    ])
    return {"answer": response.content}

sub_graph = StateGraph(SharedState)
sub_graph.add_node("analyze", analyze)
sub_graph.add_node("summarize", summarize)
sub_graph.add_edge(START, "analyze")
sub_graph.add_edge("analyze", "summarize")
sub_graph.add_edge("summarize", END)
sub_app = sub_graph.compile()

parent_graph2 = StateGraph(SharedState)
parent_graph2.add_node("sub", sub_app)
parent_graph2.add_edge(START, "sub")
parent_graph2.add_edge("sub", END)
parent_app2 = parent_graph2.compile()

result = parent_app2.invoke({"question": "Explain subgraphs in LangGraph"})
print(f"Answer: {result['answer'][:200]}...")

## 5. Multi-Level Nesting

Subgraphs can contain their own subgraphs, creating multi-level hierarchies.

In [ ]:
class NestState(TypedDict):
    data: str
    result: str

def level2_process(state: NestState) -> dict:
    return {"result": f"L2 processed: {state['data']}"}

l2_graph = StateGraph(NestState)
l2_graph.add_node("process", level2_process)
l2_graph.add_edge(START, "process")
l2_graph.add_edge("process", END)
l2_app = l2_graph.compile()

def level1_transform(state: NestState) -> dict:
    return {"data": f"L1 transformed: {state['data']}"}

l1_graph = StateGraph(NestState)
l1_graph.add_node("transform", level1_transform)
l1_graph.add_node("nested", l2_app)
l1_graph.add_edge(START, "transform")
l1_graph.add_edge("transform", "nested")
l1_graph.add_edge("nested", END)
l1_app = l1_graph.compile()

root_graph = StateGraph(NestState)
root_graph.add_node("pipeline", l1_app)
root_graph.add_edge(START, "pipeline")
root_graph.add_edge("pipeline", END)
root_app = root_graph.compile()

result = root_app.invoke({"data": "hello"})
print(f"Result: {result['result']}")

## 6. Subgraph State Isolation

Subgraph internal fields do not leak to the parent.

In [ ]:
class ParentIsoState(TypedDict):
    question: str
    answer: str

class SubIsoState(TypedDict):
    question: str
    answer: str
    internal_notes: str

def sub_node(state: SubIsoState) -> dict:
    return {
        "internal_notes": "these stay inside the subgraph",
        "answer": f"Answered: {state['question']}",
    }

sub_iso_graph = StateGraph(SubIsoState)
sub_iso_graph.add_node("work", sub_node)
sub_iso_graph.add_edge(START, "work")
sub_iso_graph.add_edge("work", END)
sub_iso_app = sub_iso_graph.compile()

def call_sub_iso(state: ParentIsoState) -> dict:
    result = sub_iso_app.invoke({"question": state["question"]})
    return {"answer": result["answer"]}

parent_iso_graph = StateGraph(ParentIsoState)
parent_iso_graph.add_node("sub", call_sub_iso)
parent_iso_graph.add_edge(START, "sub")
parent_iso_graph.add_edge("sub", END)
parent_iso_app = parent_iso_graph.compile()

result = parent_iso_app.invoke({"question": "isolation test"})
print(f"Result: {result}")
print(f"'internal_notes' in result: {'internal_notes' in result}")

## What You Learned

1. **Pattern 1** (call inside a node) handles different schemas with manual state transformation
2. **Pattern 2** (add compiled graph as node) is simpler when schemas are shared
3. Subgraphs can nest to arbitrary depth for **multi-level hierarchies**
4. Subgraph state is **isolated** — internal fields do not leak to the parent
5. Choose the pattern based on whether the parent and child share the same state schema